In [ ]:
import os  # модуль для работы с файловой системой
import sqlite3  # модуль для работы с SQLite базами данных
import xml.etree.ElementTree as ET  # модуль для парсинга XML

import pandas as pd  # модуль для работы с данными
import requests  # модуль для HTTP запросов

url = "https://iss.moex.com/iss/index.xml"  # url источника данных XML - глобальные справочники ISS
# url = "https://iss.moex.com/iss/securities/SBER/indices.xml"  # url - Список индексов в которые входит бумага SBER.
# url = "https://iss.moex.com/iss/securities/IMOEX.xml"  # url - Получить спецификацию инструмента индекс Мосбиржи.

response = requests.get(url)  # запрос данных по url

root = ET.fromstring(response.content)  # парсинг XML из ответа

dfs = {}  # словарь для хранения dataframe

for data in root.findall("./data"):  # поиск всех элементов data
    id = data.attrib["id"]  # получение атрибута id

    columns = [
        c.attrib["name"] for c in data.findall("./metadata/columns/column")
    ]  # список колонок
    df = pd.DataFrame(columns=columns)  # создание пустого dataframe с определёнными колонками
    rows = data.findall("./rows/row")  # поиск строк

    for row in rows:
        data = dict(
            zip(columns, [row.attrib.get(c) for c in columns])
        )  # словарь данных строки
        df = pd.concat(
            [df, pd.DataFrame([data])], ignore_index=True
        )  # конкатенация строк

    dfs[id] = df  # сохранение dataframe в словаре


